# Zero-Shot Baseline Test — Use Model As Designed

SmolVLM-500M scores **80.0% on ScienceQA** in the published benchmark. Our zero-shot was 0.625. Something about how we use the model is wrong. This notebook tests the model with DEFAULT settings — no custom anything.

In [1]:
!pip install -q "transformers==4.57.1" "peft==0.18.0" bitsandbytes accelerate "pandas==2.2.2" "pillow==11.3.0" tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.4 MB/s eta 0:00:00


In [2]:
import kagglehub
from pathlib import Path
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [3]:
COMP_DIR = Path(kagglehub.competition_download("pixels-to-predictions"))
print("Downloaded to:", COMP_DIR)

100%|██████████| 358M/358M [00:21<00:00, 17.5MB/s]

Extracting files...


Downloaded to: /root/.cache/kagglehub/competitions/pixels-to-predictions


In [4]:
import os, json, random, gc
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq

SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
CHOICE_LETTERS = "ABCDEFGHIJ"

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
DATA_DIR = COMP_DIR / "data"
if not DATA_DIR.exists(): DATA_DIR = COMP_DIR

val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
for df in [val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

example_name = Path(val_df.iloc[0]["image_path"]).name
matches = list(COMP_DIR.rglob(example_name))
DATA_ROOT = matches[0].parents[2]
print(f"Val: {len(val_df)} | Test: {len(test_df)}")

Val: 1048 | Test: 1008


In [6]:
# Load model with ALL defaults — no modifications
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()

# Print default settings
print("=== Default processor settings ===")
print(f"pad_token: {processor.tokenizer.pad_token}")
print(f"padding_side: {processor.tokenizer.padding_side}")
print(f"image_processor size: {processor.image_processor.size}")
if hasattr(processor.image_processor, "do_image_splitting"):
    print(f"do_image_splitting: {processor.image_processor.do_image_splitting}")
print(f"model max length: {processor.tokenizer.model_max_length}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

=== Default processor settings ===
pad_token: <|im_end|>
padding_side: right
image_processor size: {'longest_edge': 2048}
do_image_splitting: True
model max length: 8192


In [7]:
def clean_text(x):
    if pd.isna(x): return ""
    return str(x).strip()

def load_image(row):
    path = DATA_ROOT / row["image_path"]
    return Image.open(path).convert("RGB")  # native, no resize

# ============================
# PROMPT VARIANTS TO TEST
# ============================

def prompt_A_chat_simple(row):
    """Simple chat template — just question + choices"""
    choices = row["choices"]
    choices_text = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))
    question = clean_text(row.get("question", ""))

    text = f"{question}\n\n{choices_text}\n\nAnswer with just the letter."

    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": text},
    ]}]
    return processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

def prompt_B_chat_full(row):
    """Chat template with lecture + hint"""
    choices = row["choices"]
    choices_text = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))
    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))
    lecture = clean_text(row.get("lecture", ""))

    text = ""
    if lecture: text += f"Lecture: {lecture}\n\n"
    if hint: text += f"Hint: {hint}\n\n"
    text += f"Question: {question}\n\n{choices_text}\n\nAnswer with just the letter."

    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": text},
    ]}]
    return processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

def prompt_C_raw_old(row):
    """Our old raw prompt (for comparison)"""
    choices = row["choices"]
    choices_text = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))
    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))
    lecture = clean_text(row.get("lecture", ""))
    prompt = "<image>\nYou are solving a science multiple-choice question.\n"
    prompt += "Use the image and text to choose the best answer.\n"
    prompt += "Answer with ONLY the letter and the full text of the correct choice.\n\n"
    if lecture: prompt += f"Lecture: {lecture}\n\n"
    if hint: prompt += f"Hint: {hint}\n\n"
    prompt += f"Question: {question}\n\nChoices:\n{choices_text}\n\nAnswer:"
    return prompt

def prompt_D_chat_cot(row):
    """Chat template with chain-of-thought cue"""
    choices = row["choices"]
    choices_text = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))
    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))
    lecture = clean_text(row.get("lecture", ""))

    text = ""
    if lecture: text += f"Lecture: {lecture}\n\n"
    if hint: text += f"Hint: {hint}\n\n"
    text += f"Question: {question}\n\n{choices_text}\n\nThink step by step, then answer with just the letter."

    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": text},
    ]}]
    return processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

# Print a sample of each
row = val_df.iloc[0]
print("=== Prompt A (chat simple) ===")
print(prompt_A_chat_simple(row)[:200])
print("\n=== Prompt B (chat full) ===")
print(prompt_B_chat_full(row)[:200])
print("\n=== Prompt C (raw old) ===")
print(prompt_C_raw_old(row)[:200])
print("\n=== Prompt D (chat CoT) ===")
print(prompt_D_chat_cot(row)[:200])

=== Prompt A (chat simple) ===
<|im_start|>User:<image>Why might covering its eggs with its body increase the reproductive success of a snail leech? Complete the claim below that answers this question and is best supported by the p

=== Prompt B (chat full) ===
<|im_start|>User:<image>Lecture: Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in w

=== Prompt C (raw old) ===
<image>
You are solving a science multiple-choice question.
Use the image and text to choose the best answer.
Answer with ONLY the letter and the full text of the correct choice.

Lecture: Animals inc

=== Prompt D (chat CoT) ===
<|im_start|>User:<image>Lecture: Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in w


In [8]:
# ============================
# TWO INFERENCE METHODS
# ============================

def get_candidate_token_ids(tokenizer):
    cids = {}
    for letter in CHOICE_LETTERS:
        forms = [letter, " "+letter, "\n"+letter, letter+".", " "+letter+"."]
        ids = set()
        for f in forms:
            enc = tokenizer.encode(f, add_special_tokens=False)
            if enc: ids.add(enc[-1])
        cids[letter] = sorted(ids)
    return cids

candidate_ids = get_candidate_token_ids(processor.tokenizer)

def infer_logprob(df_batch, prompt_fn):
    images = [load_image(row) for _, row in df_batch.iterrows()]
    prompts = [prompt_fn(row) for _, row in df_batch.iterrows()]
    inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}
    with torch.inference_mode():
        logits = model(**inputs).logits[:, -1, :]
    log_probs = torch.log_softmax(logits, dim=-1)
    preds = []
    for i, (_, row) in enumerate(df_batch.iterrows()):
        scores = [log_probs[i, candidate_ids[CHOICE_LETTERS[ci]]].max().item()
                  for ci in range(len(row["choices"]))]
        preds.append(int(np.argmax(scores)))
    return preds

def infer_generate(df_batch, prompt_fn):
    images = [load_image(row) for _, row in df_batch.iterrows()]
    prompts = [prompt_fn(row) for _, row in df_batch.iterrows()]
    inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    input_len = inputs["input_ids"].shape[1]
    preds = []
    for i, (_, row) in enumerate(df_batch.iterrows()):
        generated = processor.tokenizer.decode(outputs[i, input_len:], skip_special_tokens=True).strip()
        letter = None
        for ch in generated:
            if ch in CHOICE_LETTERS[:len(row["choices"])]:
                letter = ch
                break
        if letter:
            preds.append(CHOICE_LETTERS.index(letter))
        else:
            preds.append(0)
    return preds

In [10]:
# ============================
# TEST ALL COMBINATIONS
# ============================
BATCH_SIZE = 2  # small batch to avoid OOM with image splitting enabled

eval_df = val_df.sample(n=200, random_state=SEED).reset_index(drop=True)

experiments = [
    ("A_chat_simple + logprob", prompt_A_chat_simple, infer_logprob),
    ("A_chat_simple + generate", prompt_A_chat_simple, infer_generate),
    ("B_chat_full + logprob", prompt_B_chat_full, infer_logprob),
    ("B_chat_full + generate", prompt_B_chat_full, infer_generate),
    ("C_raw_old + logprob", prompt_C_raw_old, infer_logprob),
    ("C_raw_old + generate", prompt_C_raw_old, infer_generate),
    ("D_chat_cot + generate", prompt_D_chat_cot, infer_generate),
]

processor.tokenizer.padding_side = "left"  # fix for generation

results = {}
for name, prompt_fn, infer_fn in experiments:
    print(f"Running: {name}...")
    preds = []
    for start in range(0, len(eval_df), BATCH_SIZE):
        batch = eval_df.iloc[start:start+BATCH_SIZE]
        try:
            p = infer_fn(batch, prompt_fn)
            preds.extend(p)
        except RuntimeError as e:
            torch.cuda.empty_cache()
            # fallback to batch=1
            for j in range(len(batch)):
                try:
                    p = infer_fn(batch.iloc[j:j+1], prompt_fn)
                    preds.extend(p)
                except RuntimeError:
                    preds.append(0)
                    torch.cuda.empty_cache()
        torch.cuda.empty_cache()

    acc = sum(p == a for p, a in zip(preds, eval_df["answer"])) / len(eval_df)
    results[name] = acc
    print(f"  {name}: {acc:.4f}")

print("\n=== RANKED RESULTS ===")
for name, acc in sorted(results.items(), key=lambda x: -x[1]):
    marker = " <<<" if acc == max(results.values()) else ""
    print(f"  {acc:.4f}  {name}{marker}")

print(f"\nOur previous zero-shot best: 0.635")
print(f"Published SmolVLM-500M ScienceQA benchmark: 0.800")

Running: A_chat_simple + logprob...
  A_chat_simple + logprob: 0.5000
Running: A_chat_simple + generate...
  A_chat_simple + generate: 0.4850
Running: B_chat_full + logprob...
  B_chat_full + logprob: 0.6050
Running: B_chat_full + generate...
  B_chat_full + generate: 0.3900
Running: C_raw_old + logprob...
  C_raw_old + logprob: 0.6250
Running: C_raw_old + generate...
  C_raw_old + generate: 0.6250
Running: D_chat_cot + generate...
  D_chat_cot + generate: 0.3300

=== RANKED RESULTS ===
  0.6250  C_raw_old + logprob <<<
  0.6250  C_raw_old + generate <<<
  0.6050  B_chat_full + logprob
  0.5000  A_chat_simple + logprob
  0.4850  A_chat_simple + generate
  0.3900  B_chat_full + generate
  0.3300  D_chat_cot + generate

Our previous zero-shot best: 0.635
Published SmolVLM-500M ScienceQA benchmark: 0.800


In [11]:
# Submit best config on test set
best_name = max(results, key=lambda k: results[k])
print(f"Best: {best_name} ({results[best_name]:.4f})")

# Find the matching prompt and inference functions
exp_map = {name: (pfn, ifn) for name, pfn, ifn in experiments}
best_prompt_fn, best_infer_fn = exp_map[best_name]

all_preds = []
for start in tqdm(range(0, len(test_df), BATCH_SIZE), desc="Test"):
    batch = test_df.iloc[start:start+BATCH_SIZE]
    try:
        p = best_infer_fn(batch, best_prompt_fn)
        all_preds.extend(p)
    except RuntimeError:
        torch.cuda.empty_cache()
        for j in range(len(batch)):
            try:
                p = best_infer_fn(batch.iloc[j:j+1], best_prompt_fn)
                all_preds.extend(p)
            except RuntimeError:
                all_preds.append(0)
                torch.cuda.empty_cache()
    torch.cuda.empty_cache()

submission = pd.DataFrame({"id": test_df["id"], "answer": all_preds})
submission.to_csv("/content/submission.csv", index=False)
submission.to_csv("/content/drive/MyDrive/submission_zeroshot_correct.csv", index=False)
print(f"Saved ({len(submission)} rows)")

from google.colab import files
files.download("/content/submission.csv")

Best: C_raw_old + logprob (0.6250)


Test:   0%|          | 0/504 [00:00<?, ?it/s]

Saved (1008 rows)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>